# Tokenizer

* [Tiktokens](https://tiktokenizer.vercel.app/)
* [Hugging Face Tokenizer docs](https://huggingface.co/docs/transformers/v5.1.0/en/fast_tokenizers)
* [Stanford Assignment1](https://github.com/stanford-cs336/assignment1-basics/blob/main/cs336_spring2025_assignment1_basics.pdf)


## Summary

#### Tokenizer: Strings <-> Tokens (indices)

Tokenization is a foundational NLP step that breaks text into smaller units (tokens) so machine learning models can process it. <br>
Why it matters?: A Transformers model expects the input to be a PyTorch or NumPy tensor. strings -> tokens -> tensors -> transformers


```

input strings: Hello, This is CiNet, NICT.
output tokens: 13225, 11, 1328, 382, 21572, 9944, 11, 478, 28594, 13

```
#### Methods
* Word-Based Tokenization
* Character-Based Tokenization
* Byte-Based Tokenization

##### 1. Word-Based Tokenization

This method splits text using delimiters(definition) such as spaces and punctuation.

**Example**

```
"Tokenization is great"
→ ["Tokenization", "is", "great"]
```
* ✅ Pros
    * Intuitive and human-readable
    * Each token usually carries semantic meaning

* ❌ Cons
    * **Enormous vocabulary**: Every unique word must be stored, often resulting in hundreds of thousands of tokens
    * **Out-of-Vocabulary (OOV) issues**: Cannot handle unseen words (e.g., typos, slang, new words)
    * **Data sparsity**: Related words like *run* and *running* are treated as completely different tokens

---

#### 2. Character-Based Tokenization

This method splits text into individual characters (letters, digits, punctuation, etc.).

**Example**

```
"Token"
→ ["T", "o", "k", "e", "n"]
```

* ✅ Pros
    * **Very small vocabulary**: About 256 possible characters in basic English
    * **No OOV problem**: Any word can be constructed from characters

* ❌ Cons
    * **Very long sequences**: Leads to high computational cost (especially for Transformers)
    * **Low semantic meaning**: Individual characters carry little meaning on their own

#### 3. Byte-Based (Byte-Level) Tokenization

This method treats text as raw UTF-8 bytes instead of characters. Even complex characters (e.g., emojis or foreign scripts) are decomposed into byte sequences.

**Example**

* A 4-byte emoji is split into 4 individual bytes.

* ✅ Pros
    * **Language-agnostic**: Works well for multilingual text
    * **Robust to rare characters and emojis**
    * **No OOV problem**

* ❌ Cons
    * Can produce long token sequences
    * Often requires additional compression (e.g., BPE) for efficiency

---

#### Summary Comparison

| Feature          | Word-Based   | Character-Based   | Byte-Based        |
| ---------------- | ------------ | ----------------- | ----------------- |
| Granularity      | High (Words) | Low (Characters)  | Very Low (Bytes)  |
| Vocabulary Size  | Very Large   | Very Small (~256) | Very Small (~256) |
| OOV Handling     | Poor         | Excellent         | Excellent         |
| Sequence Length  | Short        | Very Long         | Very Long         |
| Semantic Meaning | High         | Low               | Low               |



## Bite-Pair Encoding

### BPE Encoding (Training & Encoding)

Suppose:

**Input string**

`"the cat ate the rat"`

**Initial Vocabulary**

`{0: b' ', 1: b'a', 2: b'c', 3: b'e', 4: b'h', 5: b't', 6: b'r'}`

> cf. `bytes literal` or `b''`
> * The `b` prefix means it's a **bytes object** (not a regular string)
> * `'h'` is the character inside it
> * So `b'h'` = the byte representation of the letter "h", which is **ASCII value 104**(0x68 in hex)

---

#### Step 1 — Pre-tokenization

The input is split into space-aware chunks:

`"the cat ate the rat"  →  ['the', ' cat', ' ate', ' the', ' rat']`

Notice:

* Spaces stay attached to the following word.
* This is typical for GPT-style tokenizers.
* The leading space `' cat'` is a word-boundary signal. It tells the tokenizer:
    * `'cat'` → appears mid-word (like in "concatenate")
    * `' cat'` → appears as a standalone word (after a space)

---

#### Step 2 — Apply BPE Training (Learning Merges)

We now simulate **training**, where we repeatedly:

1. Count all adjacent token pairs
2. Merge the most frequent pair
3. Update the corpus
4. Repeat

---

##### 🔹 Iteration 1 — Count Adjacent Pairs

From the full corpus, we count all adjacent pairs:

| Pair   | Frequency |
| ------ | --------- |
| (a, t) | 3         |
| (t, h) | 2         |
| (h, e) | 2         |
| (c, a) | 1         |
| (r, a) | 1         |
| (t, e) | 1         |
| ( , c) | 1         |
| ( , a) | 1         |
| ( , t) | 1         |
| ( , r) | 1         |

* 🏆 Most Frequent Pair: `(a, t) → frequency 3`

---

**🔹 Merge Rule #1**

`a + t → at`

**Updated Corpus**

`["t","h","e"], [" ","c","at"], [" ","at","e"], ["t","h","e"], [" ","r","at"]`

**Updated Vocabulary**

`{0: b' ', 1: b'a', 2: b'c', 3: b'e', 4: b'h', 5: b't', 6: b'r', 7: b'at'}`

---

##### 🔹 Iteration 2 — Recalculate Frequencies

Now count again:

| Pair    | Frequency |
| ------- | --------- |
| (t, h)  | 2         |
| (h, e)  | 2         |
| ( , at) | 1         |
| (c, at) | 1         |
| (r, at) | 1         |
| (at, e) | 1         |
| ( , c)  | 1         |
| ( , r)  | 1         |

* 🏆 Most Frequent Pair: `(t, h) → frequency 2`

---

**🔹 Merge Rule #2**

`t + h → th`

**Updated Corpus**

`["th","e"], [" ","c","at"], [" ","at","e"], ["th","e"], [" ","r","at"]`

**Updated Vocabulary**

`{0: b' ', 1: b'a', 2: b'c', 3: b'e', 4: b'h', 5: b't', 6: b'r', 7: b'at', 8: b'th'}`

---

##### 🔹 Iteration 3 — Recalculate Frequencies

Now count again:

| Pair    | Frequency |
| ------- | --------- |
| (th, e) | 2         |
| ( , at) | 1         |
| (c, at) | 1         |
| (r, at) | 1         |
| (at, e) | 1         |
| ( , c)  | 1         |
| ( , r)  | 1         |

* 🏆 Most Frequent Pair: `(th, e) → frequency 2`

---

**🔹 Merge Rule #3**

`th + e → the`

**Updated Corpus**

`["the"], [" ","c","at"], [" ","at","e"], ["the"], [" ","r","at"]`

**Updated Vocabulary**

`{0: b' ', 1: b'a', 2: b'c', 3: b'e', 4: b'h', 5: b't', 6: b'r', 7: b'at', 8: b'th', 9: b'the'}`

---

##### 🔹 Stop Condition

Now the highest frequency is 1 for all pairs.
Training usually continues until:

* A target vocabulary size is reached
* Or no pair appears more than once

For this example, we stop here.

---

**Final Learned Merges**

1. (a, t) → at
2. (t, h) → th
3. (th, e) → the

---

**Final Vocabulary**

`{0: b' ', 1: b'a', 2: b'c', 3: b'e', 4: b'h', 5: b't', 6: b'r', 7: b'at', 8: b'th', 9: b'the'}`

---

**Now Encoding the Sentence**

Using the learned merges:

`['the', ' cat', ' ate', ' the', ' rat']`

Becomes:

`[b'the'], [b' ', b'c', b'at'], [b' ', b'at', b'e'], [b'the'], [b' ', b'r', b'at']`

Mapped to IDs:

`[9, 0, 2, 7, 0, 7, 3, 9, 0, 6, 7]`

---

##### 🔎 What This Shows

* BPE builds frequent subwords automatically.
* "the" becomes a single token.
* "at" becomes reusable for cat / ate / rat.
* Rare patterns stay broken into smaller units.

This is why BPE balances:

* Efficiency
* Small vocabulary
* No OOV problem


#### Training Code

In [ ]:
import regex as re
from collections import Counter, defaultdict
from typing import Dict, List, Tuple

# GPT-2 pre-tokenization pattern — splits raw text into chunks before BPE
# Handles: contractions ('s, 'll, 've), words, numbers, punctuation, whitespace
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def train_bpe(
    input_path: str,        # path to raw training text file
    vocab_size: int,        # target vocabulary size (e.g. 50,257 for GPT-2)
    special_tokens: List[str],  # e.g. ["<|endoftext|>"]
) -> Tuple[Dict[int, bytes], List[Tuple[bytes, bytes]]]:
    # returns:
    #   vocab  — mapping of token_id → bytes  (the final vocabulary)
    #   merges — ordered list of (bytes, bytes) merge rules learned during training

    # ── 1. Initialize Vocab ───────────────────────────────────────────────────
    # start with all 256 possible single bytes as base tokens
    # token 0 = b'\x00', token 1 = b'\x01', ..., token 255 = b'\xff'
    # this guarantees any byte sequence can be represented
    vocab: Dict[int, bytes] = {i: bytes([i]) for i in range(256)}
    next_id = 256   # next available token ID after the 256 base bytes

    # encode special tokens to bytes and add them to vocab before any merges
    # special tokens must never be split by BPE, so they get reserved IDs early
    special_token_bytes = [s.encode("utf-8") for s in special_tokens]
    for st in special_token_bytes:
        if st not in vocab.values():
            vocab[next_id] = st
            next_id += 1

    # ── 2. Build Word Frequencies ─────────────────────────────────────────────
    # read the entire training file as raw bytes
    with open(input_path, "rb") as f:
        raw_bytes = f.read()

    # split the file around special tokens so we never merge across them
    # e.g. "hello<|endoftext|>world" → [b"hello", b"<|endoftext|>", b"world"]
    segments = [raw_bytes]
    for st in special_token_bytes:
        new_segments = []
        for seg in segments:
            parts = seg.split(st)           # split this segment on the special token
            for i, part in enumerate(parts):
                if part: new_segments.append(part)              # add non-empty text chunk
                if i < len(parts) - 1: new_segments.append(st) # re-insert the special token
        segments = new_segments

    # count how often each word (as a tuple of byte-tokens) appears in the corpus
    word_freqs = Counter()
    for seg in segments:
        if seg in special_token_bytes:
            # special tokens are treated as single atomic units, never split
            word_freqs[(seg,)] += 1
        else:
            # decode segment to string so we can apply the regex pattern
            decoded = seg.decode("utf-8", errors="ignore")
            for chunk in re.findall(PAT, decoded):
                # re-encode the regex chunk to bytes, then split into individual bytes
                # e.g. " cat" → (b' ', b'c', b'a', b't')
                # this is the initial state before any merges happen
                word = tuple(bytes([b]) for b in chunk.encode("utf-8"))
                word_freqs[word] += 1

    # ── 3. Efficient State Tracking ───────────────────────────────────────────
    # store words as mutable lists (tuples can't be updated during merges)
    unique_words = [list(w) for w in word_freqs.keys()]
    counts = list(word_freqs.values())  # parallel list: counts[i] = frequency of unique_words[i]
    
    # pair_counts: how many times does each adjacent byte-pair appear across the corpus
    # e.g. {(b'c', b'a'): 150, (b'a', b't'): 200, ...}
    pair_counts = defaultdict(int)

    # pair_to_word_indices: which word indices contain a given pair
    # lets us efficiently find which words to update after a merge
    # e.g. {(b'c', b'a'): {0, 3, 7}, ...}
    pair_to_word_indices = defaultdict(set)

    # scan every word and count all adjacent pairs, weighted by word frequency
    for i, word in enumerate(unique_words):
        for pair in zip(word, word[1:]):        # zip gives all adjacent pairs
            pair_counts[pair] += counts[i]      # weight by frequency
            pair_to_word_indices[pair].add(i)   # track which word contains this pair

    merges: List[Tuple[bytes, bytes]] = []

    # ── 4. Main Merge Loop ────────────────────────────────────────────────────
    # repeat until we hit vocab_size or run out of pairs to merge
    while len(vocab) < vocab_size:
        if not pair_counts:
            break   # no more pairs left to merge
        
        # pick the most frequent pair — ties broken lexicographically by pair bytes
        best_pair = max(pair_counts.items(), key=lambda x: (x[1], x[0]))[0]
        if pair_counts[best_pair] <= 0:
            break   # remaining pairs have zero frequency, nothing useful left

        # record this merge rule in order — order matters at inference time
        merges.append(best_pair)

        # create the new merged token by concatenating the two byte sequences
        # e.g. (b'c', b'a') → b'ca'
        new_token = best_pair[0] + best_pair[1]
        vocab[next_id] = new_token
        next_id += 1

        # get all words that contain this pair, then clean up tracking dicts
        affected_word_indices = pair_to_word_indices.pop(best_pair)
        pair_counts.pop(best_pair)

        # update every word that contained the merged pair
        for i in affected_word_indices:
            word = unique_words[i]
            freq = counts[i]
            
            j = 0
            new_word = []
            while j < len(word):
                if j < len(word) - 1 and (word[j], word[j+1]) == best_pair:
                    # found the pair to merge at position j

                    # the pair (word[j-1], word[j]) no longer exists after merge
                    # decrement its count since word[j] is being consumed
                    if j > 0:
                        prev_pair = (word[j-1], word[j])
                        pair_counts[prev_pair] -= freq

                    # the pair (word[j+1], word[j+2]) no longer exists after merge
                    # decrement its count since word[j+1] is being consumed
                    if j < len(word) - 2:
                        next_pair = (word[j+1], word[j+2])
                        pair_counts[next_pair] -= freq
                    
                    # replace the two tokens with the new merged token
                    new_word.append(new_token)
                    j += 2  # skip both tokens of the pair
                else:
                    # not a merge site, keep token as-is
                    new_word.append(word[j])
                    j += 1
            
            # save the updated word back
            unique_words[i] = new_word

            # add counts for NEW pairs created by the merge
            # only pairs involving new_token are new — others were already counted
            for j in range(len(new_word) - 1):
                pair = (new_word[j], new_word[j+1])
                if new_word[j] == new_token or new_word[j+1] == new_token:
                    pair_counts[pair] += freq
                    pair_to_word_indices[pair].add(i)

        # clean up: remove any pairs whose count dropped to zero or below
        pair_counts = defaultdict(int, {k: v for k, v in pair_counts.items() if v > 0})

    return vocab, merges

In [ ]:
import os
import tempfile

# ── 1. Create a tiny corpus file ─────────────────────────────────────────────
# real BPE trains on gigabytes — here we use a tiny example to see it clearly
corpus = """
the cat sat on the mat
the cat ate the rat
a cat is a cat
<|endoftext|>
the cat sat again
"""

# write corpus to a temp file (train_bpe expects a file path)
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write(corpus)
    tmp_path = f.name

# ── 2. Run BPE training ───────────────────────────────────────────────────────
vocab_size    = 280     # 256 base bytes + special token + ~23 merges
special_tokens = ["<|endoftext|>"]

vocab, merges = train_bpe(tmp_path, vocab_size, special_tokens)
os.unlink(tmp_path)     # clean up temp file

# ── 3. Inspect vocab ──────────────────────────────────────────────────────────
print(f"Final vocab size: {len(vocab)}")
print("\nBase byte tokens (first 5):")
for i in range(5):
    print(f"  id={i:3d} → {vocab[i]}")

print("\nSpecial tokens:")
for id_, tok in vocab.items():
    if len(tok) > 4:    # special tokens are long
        print(f"  id={id_:3d} → {tok}")

print(f"\nLearned merge tokens (non-base, non-special):")
for id_, tok in vocab.items():
    if id_ >= 256 and tok not in [st.encode() for st in special_tokens]:
        print(f"  id={id_:3d} → {tok}")

# ── 4. Inspect merges ─────────────────────────────────────────────────────────
print(f"\nTotal merges learned: {len(merges)}")
print("\nMerge rules in order (first 15):")
for i, (a, b) in enumerate(merges[:15]):
    print(f"  merge {i+1:2d}: {a} + {b} → {a+b}")

# ── 5. Manual trace — watch 'cat' get merged step by step ────────────────────
print("\n── Manual trace: how ' cat' gets tokenized ──")
word = [bytes([b]) for b in " cat".encode("utf-8")]
print(f"  initial:  {word}")

for i, (a, b) in enumerate(merges):
    new_word = []
    j = 0
    while j < len(word):
        if j < len(word) - 1 and word[j] == a and word[j+1] == b:
            new_word.append(a + b)
            j += 2
        else:
            new_word.append(word[j])
            j += 1
    if new_word != word:
        print(f"  merge {i+1:2d} ({a}+{b}): {new_word}")
    word = new_word

print(f"  final:    {word}")

#### Tokenizer

In [ ]:
from __future__ import annotations

import json
import regex as re
from typing import Dict, List, Tuple, Iterable, Iterator, Any, ClassVar


# GPT-2 pre-tokenization pattern
# splits raw text into chunks before BPE is applied
# handles: contractions ('s, 'll, 've), words, numbers, punctuation, whitespace
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


def bytes_to_unicode() -> Dict[int, str]:
    # BPE merge files store tokens as unicode strings, not raw bytes
    # but not all 256 bytes are printable/safe unicode characters
    # this function creates a mapping: byte (int) → safe unicode character
    #
    # step 1: collect all "safe" printable ASCII and latin bytes
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))       
    # ! to ~   (printable ASCII)# ¡ to ¬   (latin supplement)# ® to ÿ   (more latin)
    cs = bs[:]

    # step 2: for bytes not in the safe list (e.g. control chars, null byte),
    # map them to unicode codepoints starting at 256 to avoid collisions
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)  # assign a safe unicode codepoint
            n += 1

    # result: {0: 'Ā', 1: 'ā', ..., 33: '!', ..., 255: 'ÿ'}
    return {b: chr(c) for b, c in zip(bs, cs)}


class Tokenizer:
    # class-level constant — shared across all instances, not per-instance
    pat_str: ClassVar[str] = PAT

    def __init__(
        self,
        vocab: Dict[int, bytes],        # token_id → bytes
        merges: List[Tuple[bytes, bytes]],  # ordered BPE merge rules
        special_tokens: List[str] | None = None,
    ):
        # forward lookup: token_id → bytes
        self.id_to_token = dict(vocab)

        # reverse lookup: bytes → token_id (used during encoding)
        self.token_to_id = {v: k for k, v in self.id_to_token.items()}

        # merge priority table: (bytes, bytes) → rank (lower rank = applied first)
        # e.g. {(b'c', b'a'): 0, (b'ca', b't'): 1, ...}
        self.ranks = {pair: i for i, pair in enumerate(merges)}
        
        # compile the pre-tokenization regex once at init, not on every encode call
        self.compiled_pat = re.compile(self.pat_str)
        
        # special tokens need to be matched as whole units, never split by BPE
        self.special_tokens = special_tokens or []
        if self.special_tokens:
            # sort by length descending so longer tokens match first
            # e.g. "<|endoftext|>" before "<|" if both existed
            special_pat = "|".join(
                re.escape(st) for st in sorted(self.special_tokens, key=len, reverse=True)
            )
            # wrapping in () makes re.split() keep the matched special tokens
            self.special_regex = re.compile(f"({special_pat})")
        else:
            self.special_regex = None

    @classmethod
    def from_files(
        cls,
        vocab_filepath: str,
        merges_filepath: str,
        special_tokens: List[str] | None = None
    ) -> "Tokenizer":
        # load vocab from JSON file: {"0": "!", "1": "\"", ...}
        # keys are string IDs, values are latin1-encoded token strings
        with open(vocab_filepath, "r", encoding="utf-8") as f:
            vocab_json = json.load(f)
            # convert string keys to int, and string values back to raw bytes
            # latin1 is used because it maps codepoints 0-255 directly to bytes
            vocab = {int(k): v.encode("latin1") for k, v in vocab_json.items()}

        # build the unicode→byte decoder once here (not on every line)
        byte_encoder = bytes_to_unicode()
        symbol_to_byte = {v: k for k, v in byte_encoder.items()}

        # load merge rules from text file
        # each line looks like: "ca t" meaning merge b'ca' + b't' → b'cat'
        # but stored as unicode symbols, not raw bytes
        merges = []
        with open(merges_filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#version"):
                    continue    # skip empty lines and version header
                parts = line.split()
                # convert each unicode symbol back to its original byte
                t1 = bytes([symbol_to_byte[char] for char in parts[0]])
                t2 = bytes([symbol_to_byte[char] for char in parts[1]])
                merges.append((t1, t2))

        return cls(vocab, merges, special_tokens)

    @staticmethod
    def _decode_symbol(symbol: str) -> int:
        # utility: convert a single unicode symbol back to its byte value
        # e.g. 'Ā' → 0, '!' → 33
        decoder = {v: k for k, v in bytes_to_unicode().items()}
        return decoder[symbol]

    def _apply_bpe(self, token_bytes: bytes) -> List[bytes]:
        # apply BPE merge rules to a single pre-tokenized chunk
        # e.g. b' cat' → [b' cat'] after merges, or [b' c', b'at'] if fewer merges

        # single byte can't be merged with anything
        if len(token_bytes) <= 1:
            return [token_bytes]
            
        # start with each byte as its own token
        # e.g. b' cat' → [b' ', b'c', b'a', b't']
        parts = [bytes([b]) for b in token_bytes]
        
        while len(parts) > 1:
            # collect all adjacent pairs in the current sequence
            pairs = [(parts[i], parts[i+1]) for i in range(len(parts)-1)]
            
            # find the pair with the lowest rank (= was learned earliest = highest priority)
            # pairs not in ranks get inf rank → they are never chosen
            best_pair = min(pairs, key=lambda p: self.ranks.get(p, float('inf')))
            
            # if no pair exists in our merge rules, we're done
            if best_pair not in self.ranks:
                break
                
            # apply the merge: replace ALL occurrences of best_pair in this sequence
            # (unlike training which processes left to right, here we merge all at once)
            new_parts = []
            i = 0
            while i < len(parts):
                if i < len(parts) - 1 and (parts[i], parts[i+1]) == best_pair:
                    new_parts.append(parts[i] + parts[i+1])   # merge
                    i += 2
                else:
                    new_parts.append(parts[i])                 # keep as-is
                    i += 1
            parts = new_parts

        return parts

    def encode(self, text: str) -> List[int]:
        # convert a string to a list of token IDs

        if not text:
            return []
        
        # step 1: split on special tokens first so they're never broken up by BPE
        # re.split with a capture group keeps the matched special tokens in the result
        # e.g. "hi<|endoftext|>bye" → ["hi", "<|endoftext|>", "bye"]
        if self.special_regex:
            segments = [p for p in self.special_regex.split(text) if p]
        else:
            segments = [text]

        ids = []
        for seg in segments:
            # special tokens are looked up directly, no BPE applied
            if seg in self.special_tokens:
                ids.append(self.token_to_id[seg.encode("utf-8")])
                continue
            
            # step 2: apply pre-tokenization regex to split into chunks
            # e.g. "the cat" → ["the", " cat"]
            pieces = self.compiled_pat.findall(seg)
            
            for piece in pieces:
                # step 3: encode chunk to bytes
                piece_bytes = piece.encode("utf-8")

                # step 4: apply BPE merge rules to get final tokens
                # step 5: look up each resulting byte sequence in token_to_id
                for tok in self._apply_bpe(piece_bytes):
                    ids.append(self.token_to_id[tok])

        return ids

    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        # memory-efficient encoding for large inputs (e.g. streaming a file)
        # yields one token ID at a time instead of building a full list
        # useful when the input is too large to hold in memory at once
        for text in iterable:
            for token_id in self.encode(text):
                yield token_id

    def decode(self, ids: List[int]) -> str:
        # convert token IDs back to a string
        # step 1: look up bytes for each ID and concatenate
        byte_sequence = b"".join(self.id_to_token[i] for i in ids)
        # step 2: decode the full byte sequence as UTF-8
        # errors="replace" handles cases where IDs form an incomplete UTF-8 sequence
        return byte_sequence.decode("utf-8", errors="replace")

In [ ]:
import tempfile, os, json

# ── 1. Build tiny vocab and merges files ──────────────────────────────────────
# in real usage you'd load GPT-2's vocab.json and merges.txt from HuggingFace
# here we simulate a tiny version manually

byte_enc = bytes_to_unicode()   # byte → unicode symbol map

# vocab: all 256 base bytes + a few merged tokens + special token
# stored as {str(id): unicode_symbol_string}
vocab_dict = {str(i): byte_enc[i] for i in range(256)}
vocab_dict["256"] = "<|endoftext|>"
# add a few manually merged tokens for demo
vocab_dict["257"] = byte_enc[ord(' ')] + byte_enc[ord('c')]    # ' c'
vocab_dict["258"] = byte_enc[ord('a')] + byte_enc[ord('t')]    # 'at'
vocab_dict["259"] = (
    byte_enc[ord(' ')] + byte_enc[ord('c')] +
    byte_enc[ord('a')] + byte_enc[ord('t')]                     # ' cat'
)

# merges: each line is "symbol1 symbol2"
merges_lines = [
    "#version: demo",
    f"{byte_enc[ord(' ')]} {byte_enc[ord('c')]}",   # ' ' + 'c' → ' c'
    f"{byte_enc[ord('a')]} {byte_enc[ord('t')]}",   # 'a' + 't' → 'at'
    f"{byte_enc[ord(' ')]+byte_enc[ord('c')]} "
    f"{byte_enc[ord('a')]+byte_enc[ord('t')]}",     # ' c' + 'at' → ' cat'
]

# write to temp files
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(vocab_dict, f)
    vocab_path = f.name

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write("\n".join(merges_lines))
    merges_path = f.name

# ── 2. Load tokenizer from files ──────────────────────────────────────────────
tokenizer = Tokenizer.from_files(
    vocab_filepath=vocab_path,
    merges_filepath=merges_path,
    special_tokens=["<|endoftext|>"]
)
os.unlink(vocab_path)
os.unlink(merges_path)

# ── 3. Encode some text ───────────────────────────────────────────────────────
texts = [
    "the cat sat",
    "a cat<|endoftext|>another cat",
    "cat",      # no leading space — different token than ' cat'
]

for text in texts:
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)
    print(f"text:    {repr(text)}")
    print(f"ids:     {ids}")
    print(f"tokens:  {[tokenizer.id_to_token[i] for i in ids]}")
    print(f"decoded: {repr(decoded)}")
    print()

# ── 4. encode_iterable demo ───────────────────────────────────────────────────
print("── encode_iterable (streaming) ──")
lines = ["the cat ", "sat on ", "the mat"]
token_stream = list(tokenizer.encode_iterable(lines))
print(f"input lines: {lines}")
print(f"token ids:   {token_stream}")

# ── 5. Manual _apply_bpe trace ────────────────────────────────────────────────
print("\n── Manual BPE trace: ' cat' ──")
chunk = b" cat"
parts = [bytes([b]) for b in chunk]
print(f"initial: {parts}")

while len(parts) > 1:
    pairs = [(parts[i], parts[i+1]) for i in range(len(parts)-1)]
    best = min(pairs, key=lambda p: tokenizer.ranks.get(p, float('inf')))
    if best not in tokenizer.ranks:
        break
    print(f"  apply merge {best} (rank {tokenizer.ranks[best]})")
    new_parts = []
    i = 0
    while i < len(parts):
        if i < len(parts)-1 and (parts[i], parts[i+1]) == best:
            new_parts.append(parts[i] + parts[i+1])
            i += 2
        else:
            new_parts.append(parts[i])
            i += 1
    parts = new_parts
    print(f"  result:  {parts}")

print(f"final: {parts}")